# 01 - Exploration of the 2026 survey

Shared exploration of the group. Goal: understand the raw data and justify each cleaning rule of `src/cleaning.py`.

The raw file is downloaded by the project (it is never committed).

In [1]:
import sys
sys.path.append("..")   # to import the src package of the project

import pandas as pd
from src.data import download_year
from src.config import RAW_FILES

pd.set_option("display.max_colwidth", 80)

In [2]:
download_year(2026)
raw = pd.read_csv(RAW_FILES[2026]["survey_path"], encoding="utf-8-sig", low_memory=False)
schema = pd.read_csv(RAW_FILES[2026]["schema_path"], encoding="utf-8-sig")
raw.shape

(30903, 212)

## 1. Who answered?

30,903 respondents and 212 columns. Many people have no country: let us look at them.

In [3]:
no_country = raw[raw["Country"].isna()]
with_country = raw[raw["Country"].notna()]
print("Without country:", len(no_country))
print("Average number of answers without country:", round(no_country.notna().sum(axis=1).mean(), 1))
print("Average number of answers with country:   ", round(with_country.notna().sum(axis=1).mean(), 1))
print("Salaries given by people without country:", no_country["ConvertedCompYearly"].notna().sum())

Without country: 9533
Average number of answers without country: 13.0
Average number of answers with country:    84.3
Salaries given by people without country: 0


**Decision:** the people without a country stopped the survey early (about 13 answers out of 212, no salary). We remove them.

In [4]:
raw["Country"].value_counts().head(10)

Country
United States of America                                4755
Germany                                                 1983
United Kingdom of Great Britain and Northern Ireland    1374
India                                                   1309
Canada                                                   851
France                                                   794
Netherlands                                              577
Poland                                                   527
Australia                                                522
Ukraine                                                  484
Name: count, dtype: int64

France is the 6th country of the survey and the 3rd in Europe.

## 2. Duplicates

In [5]:
answers = raw.drop(columns="ResponseId")
print("Rows with exactly the same answers to all questions:", answers.duplicated().sum())
print("Among people with a country:", with_country.drop(columns="ResponseId").duplicated().sum())

Rows with exactly the same answers to all questions: 1055
Among people with a country: 1


**Decision:** a duplicate is a row with the same answers to **all** the questions. Only 1 among people with a country.

## 3. Salaries

In [6]:
raw["ConvertedCompYearly"].describe().round(0)

count       13507.0
mean       106535.0
std        445368.0
min             1.0
25%         41372.0
50%         79610.0
75%        126347.0
max      31919350.0
Name: ConvertedCompYearly, dtype: float64

The minimum is absurd (a few dollars a year) and the maximum too. But a fixed threshold is a bad idea: look at the low salaries country by country.

In [7]:
paid = raw[raw["Employment"].isin(["Employed", "Independent contractor, freelancer, or self-employed"])]
rows = []
for country in ["France", "Germany", "United States of America", "India", "Brazil", "Nigeria"]:
    s = paid.loc[paid["Country"] == country, "ConvertedCompYearly"].dropna()
    rows.append({"country": country, "n": len(s), "median": s.median(),
                 "share below 10k $": round((s < 10_000).mean() * 100, 1)})
pd.DataFrame(rows)

,country,n,median,share below 10k $
0,France,533,62550.0,1.5
1,Germany,1235,85296.0,2.8
2,United States of America,2950,152000.0,1.6
3,India,494,18802.0,30.6
4,Brazil,269,35011.0,24.2
5,Nigeria,94,1992.0,80.9


With a fixed minimum of 10,000 $, we would remove 31% of the salaries in India and 81% in Nigeria: comparisons between countries would be wrong.

**Decision:**
1. only employed people and freelancers;
2. remove impossible values (below 1,000 $ or above 1,000,000 $ a year);
3. boxplot rule (1.5 × IQR) **country by country**, on the **log** of the salary (salaries are very skewed).

In [8]:
import numpy as np
s = paid.loc[paid["Country"] == "France", "ConvertedCompYearly"].dropna()
s = s[s.between(1_000, 1_000_000)]   # after step 2 (impossible values removed)
print("Skewness of the salary:       ", round(s.skew(), 2))
print("Skewness of the log of salary:", round(np.log10(s).skew(), 2))
print("(0 = symmetric)")

Skewness of the salary:        4.88
Skewness of the log of salary: -0.91
(0 = symmetric)


## 4. Multi-select questions

Some questions accept several answers, separated by `;`.

In [9]:
raw["LanguagesHaveWorkedWith"].dropna().head(3)

3    Bash/Shell (all shells);C++;Groovy;HTML/CSS;Java;JavaScript;Julia;Kotlin;Pow...
6                            Bash/Shell (all shells);HTML/CSS;Java;JavaScript;Kotlin
7    Assembly;Bash/Shell (all shells);C;C#;C++;Delphi;Fortran;Go;HTML/CSS;Java;Ja...
Name: LanguagesHaveWorkedWith, dtype: str

**Decision:** we keep the text and count the answers with `analysis.count_multiselect` (share of the people who answered the question).

## 5. Number of answers per question for France

Some questions have few answers: every figure of the app shows its number of respondents.

In [10]:
france = raw[raw["Country"] == "France"]
columns = ["ConvertedCompYearly", "WorkExp", "Role", "OrgSize", "WorkSetting", "LanguagesHaveWorkedWith", "AISentiment", "Gender"]
france[columns].notna().sum().sort_values(ascending=False)


Gender                     793
Role                       792
WorkExp                    755
WorkSetting                679
OrgSize                    678
ConvertedCompYearly        560
LanguagesHaveWorkedWith    515
AISentiment                490
dtype: int64

In [11]:
france["Gender"].value_counts()

Gender
Man                         673
Woman                        51
Prefer not to say            36
Non-binary                   23
Prefer to self-describe:     10
Name: count, dtype: int64

Only 51 women among the French respondents: a salary analysis by gender would not be reliable, so we do not make one.

## 6. Result of the cleaning

In [12]:
from src.cleaning import clean_survey
from src.population import paid_population, focus_country
clean = clean_survey(2026)
print(clean.shape)
print("Clean salaries:", paid_population(clean).shape[0])
print("Median salary in France (EUR):", focus_country(paid_population(clean))["SalaryEUR"].median())

(21369, 37)
Clean salaries: 11542
Median salary in France (EUR): 55000.0
